# PYTHON - Séance 2

## Partie A. Les fonctions

### A.1 Premières fonctions
On reprend le CSV de la séance 1 (`donnees/taux_EUR_USD.csv`).

In [1]:
import csv

def lire_csv(chemin):
    """Lit le CSV des taux et renvoie une liste de couples (date, taux)."""
    resultats = []
    with open(chemin, encoding="utf-8") as f:
        lecteur = csv.reader(f)
        next(lecteur)                     # on saute la ligne d'en-tête
        for ligne in lecteur:
            date = ligne[0]
            taux = float(ligne[1])        # le CSV contient du texte, on convertit en nombre
            resultats.append((date, taux))
    return resultats


def moyenne(liste_taux):
    """Renvoie la moyenne d'une liste de nombres."""
    somme = 0
    for t in liste_taux:
        somme = somme + t
    return somme / len(liste_taux)


def min_max(liste_taux):
    """Renvoie le taux minimum et le taux maximum d'une liste."""
    plus_petit = liste_taux[0]
    plus_grand = liste_taux[0]
    for t in liste_taux:
        if t < plus_petit:
            plus_petit = t
        if t > plus_grand:
            plus_grand = t
    return plus_petit, plus_grand

In [2]:
donnees = lire_csv("donnees/taux_EUR_USD.csv")

print("nombre de jours :", len(donnees))
print("trois premiers :", donnees[:3])

# on garde seulement les taux (2e élément de chaque couple)
taux = []
for date, t in donnees:
    taux.append(t)

print("moyenne :", moyenne(taux))
mini, maxi = min_max(taux)
print("min :", mini, "| max :", maxi)

nombre de jours : 171
trois premiers : [('2025-12-31', 1.175), ('2026-01-02', 1.1721), ('2026-01-05', 1.1664)]
moyenne : 1.1623684210526313
min : 1.134 | max : 1.1974


### A.2 Utiliser une fonction d'une bibliothèque

In [4]:
from statistics import mean

print("ma fonction      :", moyenne(taux))
print("statistics.mean  :", mean(taux))
print("écart            :", abs(moyenne(taux) - mean(taux)))

# on vérifie aussi min et max avec les fonctions intégrées
print(min(taux) == mini, max(taux) == maxi)

ma fonction      : 1.1623684210526313
statistics.mean  : 1.1623684210526315
écart            : 2.220446049250313e-16
True True


Une fonction sans `return` renvoie `None`.

In [5]:
def affiche_taux(t):
    print("taux :", t)          # affiche mais ne renvoie rien

resultat = affiche_taux(1.17)
print(resultat)

taux : 1.17
None


Variable locale et variable globale.

In [6]:
devise = "USD"                  # variable globale

def change_devise():
    devise = "JPY"              # variable locale, n'existe que dans la fonction
    print("dans la fonction :", devise)

change_devise()
print("en dehors        :", devise)   # la globale n'a pas changé

dans la fonction : JPY
en dehors        : USD


## Partie B. Gérer les erreurs et journaliser

On configure d'abord le fichier de log, puis on écrit la fonction d'appel à l'API avec `try/except`.

In [7]:
import logging

logging.basicConfig(
    filename="seance2.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,          # pour que la configuration marche même si on relance la cellule
)

logging.info("Début de la séance 2")

### B.1 et B.2 Appel à l'API protégé et journalisé

In [8]:
import json
import urllib.request
import urllib.error
from datetime import date


def appeler_api(url):
    """Appelle l'API et renvoie la réponse sous forme de dictionnaire, ou None en cas d'erreur."""
    logging.info(f"Appel de l'API : {url}")
    # sans cet en-tête, le serveur peut répondre 403 (accès refusé)
    requete = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

    try:
        with urllib.request.urlopen(requete, timeout=10) as r:
            texte = r.read().decode("utf-8")
        donnees = json.loads(texte)

    except urllib.error.HTTPError as e:          # le serveur répond, mais avec un code d'erreur
        logging.error(f"Erreur HTTP {e.code} pour {url}")
        print(f"Le serveur a répondu avec une erreur {e.code}.")
        return None

    except urllib.error.URLError as e:           # pas de connexion, adresse introuvable
        logging.error(f"Problème de connexion : {e.reason}")
        print("Impossible de joindre le serveur, vérifiez la connexion.")
        return None

    except json.JSONDecodeError:                 # la réponse n'est pas du JSON
        logging.error(f"Réponse illisible (pas du JSON) pour {url}")
        print("La réponse reçue est illisible.")
        return None

    else:                                        # seulement si aucune erreur
        logging.info("Réponse reçue et lue correctement")
        return donnees

    finally:                                     # dans tous les cas
        logging.info("Fin de l'appel")

**Cas normal**, le taux du jour. Si la date renvoyée n'est pas aujourd'hui (week-end, jour férié), on le signale avec un `warning`.

In [9]:
DEVISE = "USD"
BASE = "EUR"

reponse = appeler_api(f"https://api.frankfurter.app/latest?from={BASE}&to={DEVISE}")

if reponse is not None:
    print(f"Date : {reponse['date']}, 1 {BASE} = {reponse['rates'][DEVISE]} {DEVISE}")
    if reponse["date"] != str(date.today()):
        logging.warning(f"Le taux n'est pas daté d'aujourd'hui mais du {reponse['date']}")
        print("Attention, ce n'est pas le taux d'aujourd'hui.")

Date : 2026-10-07, 1 EUR = 1.1177 USD


**Cas d'erreur**, on vérifie que le programme ne s'arrête pas.

In [10]:
# 1. Mauvaise adresse sur le bon serveur -> erreur HTTP (404)
r1 = appeler_api("https://api.frankfurter.app/page_qui_n_existe_pas")
print(r1)

Le serveur a répondu avec une erreur 404.
None


In [11]:
# 2. Serveur qui n'existe pas -> problème de connexion
r2 = appeler_api("https://api.frankfurter.invalid/latest")
print(r2)

Impossible de joindre le serveur, vérifiez la connexion.
None


In [12]:
# 3. Page qui renvoie du HTML et pas du JSON -> réponse illisible
r3 = appeler_api("https://frankfurter.dev")
print(r3)

La réponse reçue est illisible.
None


On relit le fichier de log.

In [13]:
with open("seance2.log", encoding="utf-8") as f:
    print(f.read())

2026-10-07 21:46:45,705 - INFO - Début de la séance 2
2026-10-07 21:46:48,573 - INFO - Début de la séance 2
2026-10-07 21:47:02,552 - INFO - Appel de l'API : https://api.frankfurter.app/latest?from=EUR&to=USD
2026-10-07 21:47:02,719 - INFO - Réponse reçue et lue correctement
2026-10-07 21:47:02,720 - INFO - Fin de l'appel
2026-10-07 21:47:05,202 - INFO - Appel de l'API : https://api.frankfurter.app/page_qui_n_existe_pas
2026-10-07 21:47:05,336 - ERROR - Erreur HTTP 404 pour https://api.frankfurter.app/page_qui_n_existe_pas
2026-10-07 21:47:05,336 - INFO - Fin de l'appel
2026-10-07 21:47:05,915 - INFO - Appel de l'API : https://api.frankfurter.invalid/latest
2026-10-07 21:47:05,928 - ERROR - Problème de connexion : [Errno 8] nodename nor servname provided, or not known
2026-10-07 21:47:05,928 - INFO - Fin de l'appel
2026-10-07 21:47:17,680 - INFO - Appel de l'API : https://frankfurter.dev
2026-10-07 21:47:17,828 - ERROR - Réponse illisible (pas du JSON) pour https://frankfurter.dev
2026

## Partie C. Récursivité et complexité

### C.1 Fonctions récursives

In [14]:
def factorielle(n):
    """Calcule n! de façon récursive."""
    if n == 0:                         # cas de base, on arrête
        return 1
    return n * factorielle(n - 1)      # cas récursif, la fonction s'appelle elle-même


def somme_liste(liste):
    """Calcule la somme des éléments d'une liste de façon récursive."""
    if len(liste) == 0:                # cas de base, liste vide
        return 0
    return liste[0] + somme_liste(liste[1:])   # premier élément + somme du reste


print(factorielle(5))
print(somme_liste([1, 2, 3, 4]))
print(round(somme_liste(taux), 6), round(sum(taux), 6))   # même résultat que sum() sur nos taux

120
10
198.765 198.765


Profondeur trop grande, `RecursionError`.

In [15]:
import sys

print("profondeur maximale :", sys.getrecursionlimit())

try:
    factorielle(5000)
except RecursionError:
    print("RecursionError, trop d'appels imbriqués")

profondeur maximale : 3000
RecursionError, trop d'appels imbriqués


### C.2 Fibonacci, quand la récursivité coûte cher

Trois versions, récursive naïve, avec mémoïsation, itérative.

In [16]:
import time
from functools import lru_cache

nb_appels = 0      # compteur des appels (global seulement ici, pour la mesure)

def fib_naif(n):
    """Fibonacci récursif naïf."""
    global nb_appels
    nb_appels = nb_appels + 1
    if n < 2:                                  # cas de base, fib(0) = 0 et fib(1) = 1
        return n
    return fib_naif(n - 1) + fib_naif(n - 2)


@lru_cache(maxsize=None)
def fib_memo(n):
    """Fibonacci récursif avec mémoïsation (les résultats sont gardés en mémoire)."""
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)


def fib_iter(n):
    """Fibonacci avec une simple boucle."""
    a, b = 0, 1
    for i in range(n):
        a, b = b, a + b
    return a


print(fib_naif(10), fib_memo(10), fib_iter(10))   # les trois donnent 55

55 55 55


Temps de la version naïve pour des valeurs croissantes.

In [17]:
for n in [10, 15, 20, 25, 30, 32]:
    nb_appels = 0
    debut = time.perf_counter()
    fib_naif(n)
    duree = time.perf_counter() - debut
    print("n =", n, "| temps =", round(duree * 1000, 3), "ms | appels =", nb_appels)

n = 10 | temps = 0.011 ms | appels = 177
n = 15 | temps = 0.11 ms | appels = 1973
n = 20 | temps = 1.227 ms | appels = 21891
n = 25 | temps = 15.012 ms | appels = 242785
n = 30 | temps = 143.52 ms | appels = 2692537
n = 32 | temps = 356.352 ms | appels = 7049155


Le temps est multiplié par environ 1,6 à chaque fois qu'on ajoute 1 à n, il explose.

Comparaison des trois versions pour le même n.

In [18]:
def mesure(fonction, n):      # on peut donner une fonction en paramètre
    """Renvoie le temps d'exécution de fonction(n) en secondes."""
    debut = time.perf_counter()
    fonction(n)
    return time.perf_counter() - debut

n = 30
fib_memo.cache_clear()          # on vide le cache pour que la mesure soit honnête
print("naïf     :", round(mesure(fib_naif, n) * 1000, 3), "ms")
fib_memo.cache_clear()
print("mémoïsé  :", round(mesure(fib_memo, n) * 1000, 3), "ms")
print("itératif :", round(mesure(fib_iter, n) * 1000, 3), "ms")

naïf     : 134.958 ms
mémoïsé  : 0.006 ms
itératif : 0.001 ms


Pour un grand n, la version naïve est impossible, la mémoïsation marche tant qu'on ne dépasse pas la profondeur maximale, l'itératif marche toujours.

In [19]:
fib_memo.cache_clear()
print("mémoïsé  n=500  :", round(mesure(fib_memo, 500) * 1000, 3), "ms")
print("itératif n=500  :", round(mesure(fib_iter, 500) * 1000, 3), "ms")

fib_memo.cache_clear()
try:
    fib_memo(5000)
except RecursionError:
    print("mémoïsé  n=5000 : RecursionError")
print("itératif n=5000 :", round(mesure(fib_iter, 5000) * 1000, 3), "ms")

mémoïsé  n=500  : 0.126 ms
itératif n=500  : 0.015 ms
mémoïsé  n=5000 : RecursionError
itératif n=5000 : 0.343 ms


### C.3 La complexité

**Temporelle**, on compte le nombre d'opérations quand n double.

In [20]:
def une_boucle(n):
    operations = 0
    for i in range(n):
        operations = operations + 1
    return operations

def deux_boucles(n):
    operations = 0
    for i in range(n):
        for j in range(n):
            operations = operations + 1
    return operations

for n in [10, 20, 40]:
    print(f"n = {n} | une boucle : {une_boucle(n)} | deux boucles : {deux_boucles(n)}")

n = 10 | une boucle : 10 | deux boucles : 100
n = 20 | une boucle : 20 | deux boucles : 400
n = 40 | une boucle : 40 | deux boucles : 1600


In [21]:
def profondeur_fib(n, prof=1):
    """Renvoie la profondeur maximale des appels de Fibonacci naïf pour n."""
    if n < 2:                         # cas de base, on est au bout de la chaîne
        return prof
    return max(profondeur_fib(n - 1, prof + 1), profondeur_fib(n - 2, prof + 1))

for n in [10, 20, 25]:
    fib_memo.cache_clear()
    fib_memo(n)
    print("n =", n, "| profondeur max :", profondeur_fib(n), "| valeurs dans le cache :", fib_memo.cache_info().currsize)

n = 10 | profondeur max : 10 | valeurs dans le cache : 11
n = 20 | profondeur max : 20 | valeurs dans le cache : 21
n = 25 | profondeur max : 25 | valeurs dans le cache : 26


La profondeur est égale à n (la chaîne la plus longue est fib(n), fib(n-1), ..., fib(1)), et le cache garde n + 1 valeurs. Les deux grandissent avec n, c'est O(n). L'itératif reste à deux variables, c'est O(1).

Tableau récapitulatif, n = 30 pour le temps.

In [22]:
n = 30
fib_memo.cache_clear()
t_naif = mesure(fib_naif, n)
fib_memo.cache_clear()
t_memo = mesure(fib_memo, n)
t_iter = mesure(fib_iter, n)

print("naïf     |", round(t_naif * 1000, 3), "ms | pile de", n, "appels          | O(2^n) en temps, O(n) en mémoire")
print("mémoïsé  |", round(t_memo * 1000, 3), "ms | pile de", n, "+ cache de", n + 1, "| O(n) en temps, O(n) en mémoire")
print("itératif |", round(t_iter * 1000, 3), "ms | 2 variables               | O(n) en temps, O(1) en mémoire")

logging.info("Fin de la séance 2")

naïf     | 136.256 ms | pile de 30 appels          | O(2^n) en temps, O(n) en mémoire
mémoïsé  | 0.007 ms | pile de 30 + cache de 31 | O(n) en temps, O(n) en mémoire
itératif | 0.001 ms | 2 variables               | O(n) en temps, O(1) en mémoire
